In [1]:
import pandas as pd

In [2]:
# read OxCGRT
oxcgrt = pd.read_csv("raw_data\OxCGRT_AUS_latest.csv")
# recode time
oxcgrt.index = pd.to_datetime(oxcgrt["Date"], format="%Y%m%d")
# extract key column for face mask mandate
key = ["RegionName", "RegionCode", "Date", "H6M_Facial Coverings"]
oxcgrt = oxcgrt.loc[:, key]

In [3]:
# set 14 rolling days and compress each state
rolling_days = 14
oxcgrt_rolling = oxcgrt.loc[:, ["RegionName", "H6M_Facial Coverings"]].groupby("RegionName").rolling(window=rolling_days).mean()

In [4]:
# set rolling average of ≥ 3 as "the start of the mandate"
limit = 3
oxcgrt_mandates = oxcgrt_rolling[oxcgrt_rolling["H6M_Facial Coverings"]>= limit].groupby("RegionName").head(1)

In [5]:
# save start dates of mandate of each state
oxcgrt_mandates.to_csv("data/mandate_start_dates.csv")

In [6]:
# read yougov
yougov = pd.read_csv("raw_data/australia.csv",na_values=[" ", "__NA__"], keep_default_na=True, low_memory=False)

In [7]:
# count missing values and percentage
missing_info = []
for col in yougov.columns:
    missing_count = yougov[col].isna().sum()
    missing_percent = (missing_count / len(yougov)) * 100
    missing_info.append({
        'Variable Name': col,
        'Missing Count': missing_count,
        'Missing Percentage': round(missing_percent, 2)
    })

missing_value_df = pd.DataFrame(missing_info).sort_values(
    ['Missing Count', 'Variable Name'], 
    ascending=[False, True] 
)

In [8]:
# save missing value and percentage
missing_value_df.to_csv('data/missing_value_data.csv', index=False)